# Week 9 — Wednesday: Missing Data

**DATA 202 · Calvin University**

Monday, the *Titanic* model learned from 1,043 passengers. We left out **266** because a blank sat in their age, fare or port.

> 🎯 **Who goes missing from a dataset — and what should a model do about the blanks?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~7 min | 1 · Who is missing? |
| ~10 min | 2 · Three ways to go missing (SLO 09C) |
| ~6 min | 3 · Just drop them? |
| ~10 min | 4 · Fill them in: imputation (SLO 09C) |
| ~7 min | 5 · Imputation inside the pipeline (SLO 09B) |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: *titanic3*, compiled by Thomas Cason from Encyclopedia Titanica; distributed by Frank Harrell, Vanderbilt Biostatistics</sub>

---
## 1 · Who Is Missing? · ~7 min

All **1,309** passengers this time.

In [ ]:
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier

titanic = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/titanic.csv")
titanic.isna().sum()                                     # blanks in each column

`age`: 263 blanks · `cabin`: 1,014 · `embarked`: 2 · `fare`: 1. Monday's 266 = the 263 ages + 2 ports + 1 fare.

> 🎯 **Predict First:** Is a missing age equally likely in every class?

In [ ]:
titanic["age_missing"] = titanic["age"].isna()

by_class = titanic.groupby("pclass")["age_missing"].mean().reset_index()
fig = px.bar(by_class, x="pclass", y="age_missing", text_auto=".0%",
             labels={"pclass": "ticket class", "age_missing": "share with no age"},
             title="Passengers With No Recorded Age, by Class")
fig.show()

titanic.pivot_table(index="pclass", columns="age_missing", values="survived", aggfunc="mean").round(2)

> 📄 **Handout 1 — Who lost their age?** Which class lost the most ages? In each class, compare survival for passengers with and without an age. 🗣️ What story could explain both patterns?

<details><summary>Answer</summary>

* **3rd class:** 29% have no age. 1st: 12%. 2nd: 6%
* **Survival:** 41% with an age, 28% without. Even *inside* 1st class: 64% vs. 49%
* One possible story: ages were often pieced together afterwards, from documents and from survivors themselves. Passengers who died, and who traveled with fewer papers, left fewer traces. The data can't confirm it, but the blanks are clearly **not** random
</details>

---
## 2 · Three Ways to Go Missing (SLO 09C) · ~10 min

| | the blank depends on… | example |
|---|---|---|
| **MCAR** — missing completely at random | nothing: pure chance | a clerk spills ink on one line |
| **MAR** — missing at random | other columns we **can** see | more blank ages in 3rd class |
| **MNAR** — missing not at random | the **missing value itself**, or something we never recorded | people hide their age *because* of their age |

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/09/images/missing_types.png" width="520" alt="Three panels. MCAR: a grid with red missing cells scattered at random. MAR: a column with missing cells, whose missingness can be explained using the observed data. MNAR: a column whose missingness is linked to the missing values themselves, or to unobserved features.">

<sub>Image: Daily Dose of Data Science (blog.DailyDoseofDS.com)</sub>

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/09/images/manuscript.jpeg" width="520" alt="A page of an old handwritten manuscript in Spanish, with stains and faded words.">

> 💬 **Question:** On an old manuscript, water stains erased words wherever the drops fell. A later owner scraped out the passages he disagreed with. Which blanks are MCAR, and which are MNAR?

<details><summary>Answer</summary>

* **Water stains → MCAR:** where a drop fell has nothing to do with what was written there
* **Scraping → MNAR:** the content of the missing words decided which words went missing. Reading only what's left, you would never know what the owner disliked
</details>

> 📄 **Handout 2 — Name the mechanism.** Four blanks from the passenger list (one of them hypothetical). Write MCAR, MAR or MNAR, and why.

<details><summary>Answer</summary>

1. **MCAR:** a smudged line has nothing to do with the passenger
2. **MAR:** `pclass` explains it, and `pclass` is in the data
3. **MNAR:** the blank depends on the age itself. Nothing in the data can reveal it
4. **MAR:** 3rd-class tickets rarely listed a cabin, and `pclass` explains it. (It's also a column we'd drop: 77% blank)
</details>

The mechanism matters because each fix quietly assumes one of them.

---
## 3 · Just Drop Them? · ~6 min

Monday's choice: keep only the complete rows.

In [ ]:
complete = titanic.dropna(subset=["age", "fare", "embarked"])

pd.DataFrame({
    "passengers": [len(titanic), len(complete)],
    "survived": [titanic["survived"].mean(), complete["survived"].mean()],
    "3rd class": [(titanic["pclass"] == 3).mean(), (complete["pclass"] == 3).mean()],
}, index=["everyone", "complete rows"]).round(3)

<details><summary>Answer</summary>

* 3rd class: **54% → 48%** of the passengers. Survival: **38.2% → 40.7%**
* Dropping rows is safe only under **MCAR**. Here the model learns from a richer, luckier ship than the real one
</details>

And on the day a new passenger arrives with no age?

In [ ]:
X = titanic[["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]]
y = titanic["survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=1)

print(len(X_train), "training |", len(X_test), "test |", X_test.isna().any(axis=1).sum(), "test passengers with a blank")

A model trained on complete rows can't answer for those **75** test passengers: kNN can't measure a distance to a blank.

---
## 4 · Fill Them In: Imputation (SLO 09C) · ~10 min

**Imputation:** replace each blank with a guess. The simplest guess is the column's mean, median, or most frequent value, learned from the **training** rows.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean").fit(X_train[["age"]])
filled = pd.Series(imputer.transform(X_train[["age"]])[:, 0], index=X_train.index)

ages = pd.concat([X_train["age"].dropna().to_frame("age").assign(version="as recorded"),
                  filled.to_frame("age").assign(version="blanks filled with the mean")])
fig = px.histogram(ages, x="age", facet_row="version", nbins=40, title="Training Passengers' Ages, Before and After")
fig.show()

print("mean age:", round(imputer.statistics_[0], 1),
      "| standard deviation:", round(X_train["age"].std(), 1), "→", round(filled.std(), 1))

> 📄 **Handout 3 — The spike.** Circle what the imputer added. How many passengers are now exactly 30.0 years old? What happened to the spread?

<details><summary>Answer</summary>

* The **189** training passengers with no age are all 30.0 now: a spike no real ship had
* Standard deviation: **14.7 → 13.2** years. The data looks more certain than it is
* Fine for a few MCAR blanks. But our blanks are MAR on class: median age is 39 in 1st class and 24 in 3rd. One mean fits neither
</details>

Better guesses use the columns we *can* see:

| | fills a blank with… | `sklearn.impute` |
|---|---|---|
| mean / median / most frequent | one value for everyone | `SimpleImputer` |
| similar passengers | the average of the k most similar rows | `KNNImputer` (scale first: Monday!) |
| a model's prediction | a regression from the other columns, repeated | `IterativeImputer` |

Or keep the blank as information: `add_indicator=True` adds a column that is 1 when the value was missing.

In [ ]:
SimpleImputer(strategy="median", add_indicator=True).fit(X_train[["age"]]).transform(X_test[["age"]].head(8))

> "The simplest eigenstatus of all is an indicator of mere presence or absence, observed or unobserved, on some dimension. If you are not included, you cannot be measured or assessed."
> — Fourcade & Healy, *The Ordinal Society*, ch. 3

> 💬 **Question:** With the indicator, the model can learn "no age recorded → less likely to survive." Is that a fact about the passengers, or about the people who kept the records? Should the model use it?

<details><summary>Answer</summary>

* It's both. The blank traces back to who left papers, who was asked afterwards, and who could still answer
* For predicting it may help: the blank carries real information when it's MNAR
* But the model then scores people partly on whether an institution recorded them. Today that might be a thin credit file or a missing medical history
</details>

---
## 5 · Imputation Inside the Pipeline (SLO 09B) · ~7 min

Monday's pipeline, with one more step in each branch. The imputer learns its fill values from the training folds only, like the scaler.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric, categorical = ["pclass", "age", "sibsp", "parch", "fare"], ["sex", "embarked"]

def titanic_model(imputer):
    """Monday's pipeline (kNN, k = 25), with an imputer in front of the scaler."""
    num_steps = [imputer, StandardScaler()] if imputer is not None else [StandardScaler()]
    prep = ColumnTransformer([
        ("num", make_pipeline(*num_steps), numeric),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(drop="if_binary")), categorical),
    ])
    return Pipeline([("prep", prep), ("knn", KNeighborsClassifier(n_neighbors=25))])

titanic_model(SimpleImputer(strategy="median", add_indicator=True))

In [ ]:
complete_rows = X_train.notna().all(axis=1)
rows = [{"strategy": "drop incomplete rows", "training rows": complete_rows.sum(),
         "CV accuracy": cross_val_score(titanic_model(None), X_train[complete_rows], y_train[complete_rows], cv=5).mean()}]
for name, imputer in [("mean", SimpleImputer(strategy="mean")),
                      ("median", SimpleImputer(strategy="median")),
                      ("median + indicator", SimpleImputer(strategy="median", add_indicator=True)),
                      ("k nearest passengers", KNNImputer(n_neighbors=5))]:
    rows.append({"strategy": name, "training rows": len(X_train),
                 "CV accuracy": cross_val_score(titanic_model(imputer), X_train, y_train, cv=5).mean()})
pd.DataFrame(rows).round(3)

<details><summary>Answer</summary>

* All five land between **78.1% and 78.7%**. The 5 folds disagree by about 4 points, so these differences are noise
* What the strategies *do* differ on:
  * **who is included:** dropping loses 191 training passengers, mostly 3rd class, and can't answer for 75 test passengers
  * **what they assume:** mean or median treats the blanks as MCAR; the indicator treats the blank itself as a signal
* Choose by the mechanism you believe, then check the score. Not the other way around
</details>

In [ ]:
final = titanic_model(SimpleImputer(strategy="median", add_indicator=True)).fit(X_train, y_train)
no_age = X_test["age"].isna()
print("test accuracy:", round(final.score(X_test, y_test), 3),
      "| on the", no_age.sum(), "test passengers with no age:", round(final.score(X_test[no_age], y_test[no_age]), 3))
print("lazy model, 'everyone died':", round((y_test == 0).mean(), 3))

> 📌 **One split:** 328 test passengers, 74 of them with no age. Every passenger got an answer.

---
### Wrap-up

1. **Look before you fill:** count the blanks, and ask *who* they belong to
2. **MCAR / MAR / MNAR:** chance, something we see, or the missing value itself (or something unseen)
3. **Dropping rows** changes who is in the data unless the blanks are MCAR, and leaves blank passengers without an answer
4. **Imputation** goes **inside the pipeline**, learned from training rows only. A missing-indicator column keeps the blank as information, for better or worse 📌